### SQL Databases

In [1]:
##  create a sample SQLite database 
import sqlite3
import os

os.makedirs("data/databases", exist_ok=True)

In [2]:
## create a sample SQLite database
conn = sqlite3.connect("data/databases/company.db")
cursor = conn.cursor()

In [3]:
# Create tables
cursor.execute("""
CREATE TABLE IF NOT EXISTS employees
(
    id INTEGER PRIMARY KEY,
    name TEXT,
    role TEXT,
    department TEXT,
    salary REAL
)
""")


In [4]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS projects
(
    id INTEGER PRIMARY KEY,
    name TEXT,
    status TEXT,
    budget REAL,
    lead_id INTEGER
)
""")

In [5]:
# Insert sample data

employees = [
    (1, 'John Doe', 'Senior Developer', 'Engineering', 95000),
    (2, 'Jane Smith', 'Data Scientist', 'Analytics', 105000),
    (3, 'Mike Johnson', 'Product Manager', 'Product', 110000),
    (4, 'Sarah Williams', 'DevOps Engineer', 'Engineering', 98000)
]

projects = [
    (1, 'RAG Implementation', 'Active', 150000, 1),
    (2, 'Data Pipeline', 'Completed', 80000, 2),
    (3, 'Customer Portal', 'Planning', 200000, 3),
    (4, 'ML Platform', 'Active', 250000, 2)
]

In [7]:
cursor.executemany("INSERT OR REPLACE INTO employees VALUES (?, ?, ?, ?, ?)", employees)
cursor.executemany("INSERT OR REPLACE INTO projects VALUES (?, ?, ?, ?, ?)", projects)

In [8]:
cursor.execute("SELECT * FROM employees")

In [9]:
conn.commit()
conn.close()

### Database Content Extraction

In [11]:
from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader


In [13]:
# method 1 SQL Database utility
db = SQLDatabase.from_uri("sqlite:///data/databases/company.db")

#get the Database info
print(f"Tables in the database: {db.get_usable_table_names()}")
print(f"\nTable DDL:")
print(db.get_table_info())


Tables in the database: ['employees', 'projects']

Table DDL:

CREATE TABLE employees (
	id INTEGER, 
	name TEXT, 
	role TEXT, 
	department TEXT, 
	salary REAL, 
	PRIMARY KEY (id)
)

/*
3 rows from employees table:
id	name	role	department	salary
1	John Doe	Senior Developer	Engineering	95000.0
2	Jane Smith	Data Scientist	Analytics	105000.0
3	Mike Johnson	Product Manager	Product	110000.0
*/


CREATE TABLE projects (
	id INTEGER, 
	name TEXT, 
	status TEXT, 
	budget REAL, 
	lead_id INTEGER, 
	PRIMARY KEY (id)
)

/*
3 rows from projects table:
id	name	status	budget	lead_id
1	RAG Implementation	Active	150000.0	1
2	Data Pipeline	Completed	80000.0	2
3	Customer Portal	Planning	200000.0	3
*/


In [18]:
# Method 2 : Custom SQL to Documents conversion

from typing import List
from langchain_core.documents import Document
import sqlite3


print("\n\n--- Custom SQL Processing ---\n")


def sql_to_documents(db_path: str) -> List[Document]:

    # Connect database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Empty documents list
    documents = []

    # Get all table names
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    # Loop through tables
    for table in tables:

        table_name = table[0]

        # Get table schema
        cursor.execute(f"PRAGMA table_info({table_name});")
        columns = cursor.fetchall()

        column_names = [col[1] for col in columns]

        # Get table data
        cursor.execute(f"SELECT * FROM {table_name};")
        rows = cursor.fetchall()

        # Create table overview content
        table_content = f"""
        Table Name: {table_name}

        Columns: {', '.join(column_names)}

        Total Records: {len(rows)}

        Sample Data:
        """

        # Add sample rows
        for row in rows[:5]:

            record = dict(zip(column_names, row))

            table_content += f"\n{record}"

        # Create LangChain document
        doc = Document(
            page_content=table_content,

            metadata={
                "source": db_path,
                "table_name": table_name,
                "num_records": len(rows),
                "database_type": "sqlite"
            }
        )

        # Add document into list
        documents.append(doc)

    # Close connection
    conn.close()

    # Return documents
    return documents



--- Custom SQL Processing ---



In [21]:
sql_to_documents("data/databases/company.db")

[Document(metadata={'source': 'data/databases/company.db', 'table_name': 'employees', 'num_records': 4, 'database_type': 'sqlite'}, page_content="\n        Table Name: employees\n\n        Columns: id, name, role, department, salary\n\n        Total Records: 4\n\n        Sample Data:\n        \n{'id': 1, 'name': 'John Doe', 'role': 'Senior Developer', 'department': 'Engineering', 'salary': 95000.0}\n{'id': 2, 'name': 'Jane Smith', 'role': 'Data Scientist', 'department': 'Analytics', 'salary': 105000.0}\n{'id': 3, 'name': 'Mike Johnson', 'role': 'Product Manager', 'department': 'Product', 'salary': 110000.0}\n{'id': 4, 'name': 'Sarah Williams', 'role': 'DevOps Engineer', 'department': 'Engineering', 'salary': 98000.0}"),
 Document(metadata={'source': 'data/databases/company.db', 'table_name': 'projects', 'num_records': 4, 'database_type': 'sqlite'}, page_content="\n        Table Name: projects\n\n        Columns: id, name, status, budget, lead_id\n\n        Total Records: 4\n\n        S